# PromptedCRE — Autoresearch Market Refresh
**What this does:** Searches for fresh industrial RE market data, updates `MARKET-CONTEXT.md`, and pushes it back to GitHub automatically.

**Run this:** Weekly (every Monday morning) or any time you want current data.

---
### Setup
Before running, fill in your API keys in the cell below.

In [ ]:
# ============================================================
# CONFIGURATION — Fill these in before running
# ============================================================

ANTHROPIC_API_KEY = ""   # Get from console.anthropic.com
BRAVE_API_KEY = ""       # Get from api.search.brave.com (free tier works)
GITHUB_TOKEN = ""        # GitHub > Settings > Developer Settings > Personal Access Tokens
GITHUB_REPO = "dueriii/v0-premium-industrial-landing-page"  # your repo
GITHUB_BRANCH = "main"
MARKET_CONTEXT_PATH = "repo/MARKET-CONTEXT.md"  # path inside repo

# Markets to research
MARKETS = [
    "Houston Texas industrial real estate",
    "San Antonio Texas industrial real estate",
    "Schertz New Braunfels I-35 corridor industrial",
    "Texas industrial cap rates",
    "small bay industrial market trends",
]

In [ ]:
# Install dependencies
!pip install anthropic requests -q

In [ ]:
import anthropic
import requests
import json
import base64
from datetime import datetime

today = datetime.now().strftime("%Y-%m-%d")
print(f"Starting market refresh — {today}")

## Step 1 — Search for Fresh Market Data

In [ ]:
def brave_search(query, count=5):
    """Search Brave for fresh market data."""
    headers = {
        "Accept": "application/json",
        "Accept-Encoding": "gzip",
        "X-Subscription-Token": BRAVE_API_KEY
    }
    params = {
        "q": query + " 2025 2026 market report",
        "count": count,
        "freshness": "pm"  # past month
    }
    resp = requests.get(
        "https://api.search.brave.com/res/v1/web/search",
        headers=headers,
        params=params
    )
    if resp.status_code != 200:
        print(f"  Search error for '{query}': {resp.status_code}")
        return []
    results = resp.json().get("web", {}).get("results", [])
    return [{"title": r["title"], "url": r["url"], "description": r.get("description", "")} for r in results]


print("Searching for market data...\n")
all_results = {}

for market in MARKETS:
    print(f"  Searching: {market}")
    results = brave_search(market)
    all_results[market] = results
    print(f"    Found {len(results)} results")

print("\nSearch complete.")

## Step 2 — Load Current MARKET-CONTEXT.md from GitHub

In [ ]:
def github_get_file(repo, path, branch, token):
    """Fetch a file from GitHub and return its content + SHA."""
    url = f"https://api.github.com/repos/{repo}/contents/{path}"
    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github+json"
    }
    resp = requests.get(url, headers=headers, params={"ref": branch})
    resp.raise_for_status()
    data = resp.json()
    content = base64.b64decode(data["content"]).decode("utf-8")
    return content, data["sha"]


print("Loading current MARKET-CONTEXT.md from GitHub...")
current_context, file_sha = github_get_file(
    GITHUB_REPO, MARKET_CONTEXT_PATH, GITHUB_BRANCH, GITHUB_TOKEN
)
print(f"Loaded ({len(current_context)} chars, SHA: {file_sha[:8]}...)")

## Step 3 — Run the Autoresearch Loop with Claude

In [ ]:
def format_search_results(results_dict):
    """Format search results into a readable digest for Claude."""
    lines = []
    for query, results in results_dict.items():
        lines.append(f"\n### Search: {query}")
        for r in results:
            lines.append(f"- **{r['title']}** ({r['url']})")
            if r['description']:
                lines.append(f"  {r['description'][:300]}")
    return "\n".join(lines)


search_digest = format_search_results(all_results)

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

SYSTEM_PROMPT = """You are an industrial real estate market researcher and data analyst.
You specialize in the Houston and San Antonio Texas industrial markets.
Your job is to synthesize fresh market intelligence and produce accurate, specific, 
actionable market context for real estate professionals.
Always cite specific numbers when available. Flag when data is estimated vs. confirmed.
Today's date: """ + today

USER_PROMPT = f"""I need you to update my industrial real estate market context file.

Here is the CURRENT market context file:
---
{current_context}
---

Here is fresh research I just gathered from the web:
---
{search_digest}
---

Your task:
1. Analyze the fresh research for any new data points on:
   - Houston industrial vacancy rates
   - Houston asking rents by submarket
   - San Antonio / I-35 corridor market conditions
   - Industrial cap rates in Texas
   - Construction pipeline and new deliveries
   - National industrial benchmarks
   - Any significant market shifts since the current file was written

2. Produce an UPDATED MARKET-CONTEXT.md that:
   - Keeps the same structure and format as the current file
   - Updates any numbers that have new data supporting a change
   - Adds any important new data points not currently in the file
   - Updates the "Last updated" date to today ({today})
   - Notes at the bottom what changed and why (in a ## WHAT CHANGED section)
   - If no meaningful new data was found, keeps the file as-is and notes that

3. After the updated file, list any WORKFLOW FLAGS:
   - Specific workflow files (01-intake through 08-memo-output) that may need updating
     based on market changes
   - Be specific: "Workflow 02 search filters — update budget reality check benchmarks"

Output the complete updated MARKET-CONTEXT.md file first (starting with # MARKET-CONTEXT.md),
then the WORKFLOW FLAGS section.
"""

print("Running autoresearch loop with Claude...")
message = client.messages.create(
    model="claude-opus-4-5",
    max_tokens=4096,
    messages=[{"role": "user", "content": USER_PROMPT}],
    system=SYSTEM_PROMPT
)

full_response = message.content[0].text
print("Claude response received.")
print(f"Response length: {len(full_response)} chars")

## Step 4 — Parse Response & Preview

In [ ]:
# Split the response into updated context + workflow flags
if "## WORKFLOW FLAGS" in full_response:
    parts = full_response.split("## WORKFLOW FLAGS")
    updated_context = parts[0].strip()
    workflow_flags = "## WORKFLOW FLAGS\n" + parts[1].strip()
else:
    updated_context = full_response.strip()
    workflow_flags = "No workflow flags identified."

print("=" * 60)
print("UPDATED MARKET-CONTEXT.md (preview — first 2000 chars):")
print("=" * 60)
print(updated_context[:2000])
print("\n" + "=" * 60)
print("WORKFLOW FLAGS:")
print("=" * 60)
print(workflow_flags)

## Step 5 — Push Updated File to GitHub
**Review the preview above before running this cell.**

In [ ]:
def github_update_file(repo, path, branch, token, content, sha, commit_message):
    """Update a file on GitHub."""
    url = f"https://api.github.com/repos/{repo}/contents/{path}"
    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github+json"
    }
    encoded = base64.b64encode(content.encode("utf-8")).decode("utf-8")
    payload = {
        "message": commit_message,
        "content": encoded,
        "sha": sha,
        "branch": branch
    }
    resp = requests.put(url, headers=headers, json=payload)
    resp.raise_for_status()
    return resp.json()


commit_msg = f"autoresearch: update MARKET-CONTEXT.md — {today}"

result = github_update_file(
    GITHUB_REPO,
    MARKET_CONTEXT_PATH,
    GITHUB_BRANCH,
    GITHUB_TOKEN,
    updated_context,
    file_sha,
    commit_msg
)

print(f"✅ Pushed to GitHub!")
print(f"Commit: {result['commit']['sha'][:8]}")
print(f"URL: {result['commit']['html_url']}")

print("\n--- WORKFLOW FLAGS (action items for you) ---")
print(workflow_flags)

## Done.
**What just happened:**
1. Searched the web for fresh Houston, SA, and Texas industrial market data
2. Claude compared it to your existing MARKET-CONTEXT.md
3. Updated the file with any new numbers
4. Pushed the update to GitHub
5. Flagged any workflow files that may need updating

**Run this notebook weekly** (every Monday) to keep your repo current.

---
### Optional: Schedule in Google Colab
Google Colab doesn't have native scheduling, but you can:
- Use **GitHub Actions** (free) to run this notebook weekly automatically
- Set a calendar reminder to run it manually each Monday
- Use **Colab's scheduled notebooks** feature (Colab Pro)
